In [1]:
"""
Constant-size “zoom-in” Rolling Precision QUBO (with backtracking + dynamic penalties)
====================================================================================

This implements the *constant-QUBO-size* rolling-precision idea:

- FIX the number of bits (J) for every continuous/slack variable -> QUBO size stays constant.
- Iteratively shrink the variable bounds (“zoom in”) around the incumbent solution.
- Use backtracking (revert to previous bounds) if a zoom attempt fails to improve.
- Use dynamic penalty weights for BUILDING the QUBO that scale with the current box width,
  but evaluate improvement using a fixed feasibility/objective criterion (so comparisons stay fair).

Large MINLP instance (“alan” from MINLPLib):
-------------------------------------------
Variables:
  x1..x4 in [0,1] (continuous)
  b6..b9 in {0,1} (binary)

Constraints:
  e1: x1 + x2 + x3 + x4 = 1
  e2: 8x1 + 9x2 + 12x3 + 7x4 = 10
  link: x_i <= b_i  (i=1..4)
  card: b6 + b7 + b8 + b9 <= 3

Objective:
  minimize  4 x1^2 + 6 x1 x2 - 2 x1 x3 + 6 x2^2 + 2 x2 x3 + 10 x3^2

Slack-penalty reformulations used here (consistent with your earlier style):
  link:  x_i - b_i + s_i = 0,  with s_i in [0,1]
  card:  (b6+b7+b8+b9) + sc = 3, with sc in {0,1,2,3} (2 binary bits)

Encoding:
  - x_i and s_i use SBE decimal encoding (weights 1,2,3,3 plus tail bit), FIXED J
  - sc is exact 2-bit integer encoding
  - b are native binaries

Dependencies:
  pip install dimod dwave-neal
"""

import time
from dataclasses import dataclass
from typing import Dict, Tuple, List, Any, Optional

import dimod
from neal import SimulatedAnnealingSampler

# ----------------------------
# SBE encoding (constant size)
# ----------------------------

DEC_WEIGHTS = (1, 2, 3, 3)  # SBE digit weights: represent any digit 0..9 using 4 bits


def sbe_affine(var: str, J: int, L: float, U: float) -> Tuple[float, Dict[str, float]]:
    """
    Build an affine form x = c + sum_i a_i z_i for SBE-decimal encoding on [L,U].
    Bits: z_{var}_{j}_{k} for j=1..J, k=1..4 plus tail bit z_{var}_tail_J{J}.
    """
    if J < 1:
        raise ValueError("SBE requires J >= 1")
    if U < L:
        raise ValueError("Bad bounds: U < L")

    c = L
    width = (U - L)
    coeffs: Dict[str, float] = {}

    for j in range(1, J + 1):
        place = 10 ** (-j)
        for k, w in enumerate(DEC_WEIGHTS, start=1):
            b = f"z_{var}_{j}_{k}"
            coeffs[b] = coeffs.get(b, 0.0) + width * place * w

    tail = f"z_{var}_tail_J{J}"
    coeffs[tail] = coeffs.get(tail, 0.0) + width * (10 ** (-J))

    return c, coeffs


def decode_affine(sample: Dict[str, int], c: float, coeffs: Dict[str, float]) -> float:
    val = c
    for b, a in coeffs.items():
        val += a * float(sample.get(b, 0))
    return val


def normalized_coord(x: float, L: float, U: float) -> float:
    """xhat in [0,1] such that x = L + (U-L)*xhat."""
    w = (U - L)
    if w <= 0:
        return 0.0
    xhat = (x - L) / w
    if xhat < 0:
        return 0.0
    if xhat > 1:
        return 1.0
    return xhat


# ----------------------------
# QUBO helpers
# ----------------------------

def add_linear(Qlin: Dict[str, float], v: str, w: float):
    Qlin[v] = Qlin.get(v, 0.0) + w


def add_quad(Qquad: Dict[Tuple[str, str], float], u: str, v: str, w: float):
    if u == v:
        # push diagonal to linear using z^2=z (dimod also allows diag but keep clean)
        raise ValueError("Use add_linear for diagonal terms.")
    a, b = (u, v) if u < v else (v, u)
    Qquad[(a, b)] = Qquad.get((a, b), 0.0) + w


def add_square_of_affine(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    c0: float,
    coeffs: Dict[str, float],
    weight: float,
):
    """
    Add weight * (c0 + sum_i a_i z_i)^2
    Using z_i^2 = z_i.
    """
    offset_ref[0] += weight * (c0 * c0)

    items = list(coeffs.items())
    # linear terms
    for bi, ai in items:
        add_linear(Qlin, bi, weight * (ai * ai + 2.0 * c0 * ai))

    # quadratic terms
    for i in range(len(items)):
        bi, ai = items[i]
        for j in range(i + 1, len(items)):
            bj, aj = items[j]
            add_quad(Qquad, bi, bj, weight * (2.0 * ai * aj))


def add_product_of_affines(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    c1: float,
    a1: Dict[str, float],
    c2: float,
    a2: Dict[str, float],
    weight: float,
):
    """
    Add weight * (c1 + sum_i a1_i z_i) * (c2 + sum_j a2_j z_j)
    Uses z^2=z when i=j (rare here because different vars use different bit names).
    """
    offset_ref[0] += weight * (c1 * c2)

    # linear from c1 * a2 + c2 * a1
    for b, a in a2.items():
        add_linear(Qlin, b, weight * (c1 * a))
    for b, a in a1.items():
        add_linear(Qlin, b, weight * (c2 * a))

    # quadratic from a1_i * a2_j
    for bi, ai in a1.items():
        for bj, aj in a2.items():
            if bi == bj:
                # z^2=z
                add_linear(Qlin, bi, weight * (ai * aj))
            else:
                add_quad(Qquad, bi, bj, weight * (ai * aj))


def bqm_stats(bqm: dimod.BinaryQuadraticModel) -> Dict[str, int]:
    return {
        "n_vars": len(bqm.variables),
        "n_lin": len(bqm.linear),
        "n_quad": len(bqm.quadratic),
    }


def rescale_bqm_inplace(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    target_max_abs: float = 10.0,
) -> float:
    """
    Optional numeric stabilization:
    If max |bias| is huge, scale everything down.
    Returns scale factor applied (1.0 means no scaling).
    """
    max_abs = 0.0
    for v in Qlin.values():
        max_abs = max(max_abs, abs(v))
    for v in Qquad.values():
        max_abs = max(max_abs, abs(v))
    max_abs = max(max_abs, abs(offset_ref[0]))

    if max_abs <= target_max_abs or max_abs == 0.0:
        return 1.0

    s = max_abs / target_max_abs
    for k in list(Qlin.keys()):
        Qlin[k] /= s
    for k in list(Qquad.keys()):
        Qquad[k] /= s
    offset_ref[0] /= s
    return s


# ----------------------------
# Problem-specific (alan) model
# ----------------------------

@dataclass
class SAOptions:
    num_reads: int = 300
    sweeps: int = 4000
    seed: Optional[int] = 13


@dataclass
class PenaltyConfig:
    # Base penalties (these are scaled dynamically for BUILDING, see code)
    lam_e1: float = 200.0
    lam_e2: float = 200.0
    lam_link: float = 200.0
    lam_card: float = 200.0

    # Dynamic scaling control
    dynamic: bool = True
    lam_cap: float = 1e6  # cap to avoid gigantic coefficients
    eps: float = 1e-12    # avoid division by zero


@dataclass
class Box:
    # bounds for x_i and link slacks s_i
    x_bounds: Dict[str, Tuple[float, float]]
    s_bounds: Dict[str, Tuple[float, float]]  # s1..s4 for link constraints


def solve_alan_at_box(
    box: Box,
    Jx: int,
    Js: int,
    penalties: PenaltyConfig,
    sa: SAOptions,
    rescale_target: float = 10.0,
) -> Dict[str, Any]:
    """
    Build and solve BQM for the alan instance at the given bounds box.
    Returns decoded x, b, slacks, objective, residuals, BQM stats, runtime, etc.
    """

    # --- dynamic penalty scaling based on current width ---
    # Idea: when widths shrink, bit coefficients shrink linearly with width,
    # so quadratic constraint penalties should scale like 1/width^2 to keep "strength".
    def width_scale_x() -> float:
        return max((U - L) for (L, U) in box.x_bounds.values())

    def width_scale_s() -> float:
        return max((U - L) for (L, U) in box.s_bounds.values())

    wx = width_scale_x()
    ws = width_scale_s()
    wref = max(wx, ws, penalties.eps)

    if penalties.dynamic:
        scale = (1.0 / (wref * wref + penalties.eps))
    else:
        scale = 1.0

    lam_e1 = min(penalties.lam_cap, penalties.lam_e1 * scale)
    lam_e2 = min(penalties.lam_cap, penalties.lam_e2 * scale)
    lam_link = min(penalties.lam_cap, penalties.lam_link * scale)
    lam_card = min(penalties.lam_cap, penalties.lam_card * scale)

    # --- build affine encodings ---
    # x variables
    cx, ax = {}, {}
    for xi, (L, U) in box.x_bounds.items():
        c, a = sbe_affine(xi, Jx, L, U)
        cx[xi] = c
        ax[xi] = a

    # link slacks s1..s4 in [0,1] but zoomed inside that interval
    cs, as_ = {}, {}
    for si, (L, U) in box.s_bounds.items():
        c, a = sbe_affine(si, Js, L, U)
        cs[si] = c
        as_[si] = a

    # binaries b6..b9
    bnames = ["b6", "b7", "b8", "b9"]

    # cardinality slack sc in {0,1,2,3} exact 2 bits:
    # sc = sc0 + 2 sc1
    sc0, sc1 = "sc0", "sc1"

    # --- Build QUBO ---
    Qlin: Dict[str, float] = {}
    Qquad: Dict[Tuple[str, str], float] = {}
    offset = [0.0]

    # ---- objective (quadratic in x1,x2,x3 only) ----
    # f = 4 x1^2 + 6 x1 x2 - 2 x1 x3 + 6 x2^2 + 2 x2 x3 + 10 x3^2
    add_square_of_affine(Qlin, Qquad, offset, cx["x1"], ax["x1"], weight=4.0)
    add_product_of_affines(Qlin, Qquad, offset, cx["x1"], ax["x1"], cx["x2"], ax["x2"], weight=6.0)
    add_product_of_affines(Qlin, Qquad, offset, cx["x1"], ax["x1"], cx["x3"], ax["x3"], weight=-2.0)
    add_square_of_affine(Qlin, Qquad, offset, cx["x2"], ax["x2"], weight=6.0)
    add_product_of_affines(Qlin, Qquad, offset, cx["x2"], ax["x2"], cx["x3"], ax["x3"], weight=2.0)
    add_square_of_affine(Qlin, Qquad, offset, cx["x3"], ax["x3"], weight=10.0)

    # ---- constraints ----

    # e1: x1 + x2 + x3 + x4 = 1
    g1_0 = (cx["x1"] + cx["x2"] + cx["x3"] + cx["x4"] - 1.0)
    g1: Dict[str, float] = {}
    for xi in ["x1", "x2", "x3", "x4"]:
        for b, a in ax[xi].items():
            g1[b] = g1.get(b, 0.0) + a
    add_square_of_affine(Qlin, Qquad, offset, g1_0, g1, weight=lam_e1)

    # e2: 8x1 + 9x2 + 12x3 + 7x4 = 10
    g2_0 = (8.0 * cx["x1"] + 9.0 * cx["x2"] + 12.0 * cx["x3"] + 7.0 * cx["x4"] - 10.0)
    g2: Dict[str, float] = {}
    weights = {"x1": 8.0, "x2": 9.0, "x3": 12.0, "x4": 7.0}
    for xi in ["x1", "x2", "x3", "x4"]:
        wi = weights[xi]
        for b, a in ax[xi].items():
            g2[b] = g2.get(b, 0.0) + wi * a
    add_square_of_affine(Qlin, Qquad, offset, g2_0, g2, weight=lam_e2)

    # link constraints: x_i <= b_{5+i}  -> x_i - b + s_i = 0, s_i in [0,1]
    # Map: x1<=b6 uses slack s1, x2<=b7 uses s2, x3<=b8 uses s3, x4<=b9 uses s4
    link_map = [("x1", "b6", "s1"), ("x2", "b7", "s2"), ("x3", "b8", "s3"), ("x4", "b9", "s4")]
    for xi, bi, si in link_map:
        r0 = cx[xi] + cs[si]  # (x_i + s_i) - b_i
        r: Dict[str, float] = {}
        # x bits
        for b, a in ax[xi].items():
            r[b] = r.get(b, 0.0) + a
        # slack bits
        for b, a in as_[si].items():
            r[b] = r.get(b, 0.0) + a
        # binary term: -1 * b_i
        r[bi] = r.get(bi, 0.0) - 1.0

        add_square_of_affine(Qlin, Qquad, offset, r0, r, weight=lam_link)

    # cardinality: b6+b7+b8+b9 <= 3  -> (b_sum + sc - 3) = 0 with sc in {0,1,2,3}
    # sc = sc0 + 2 sc1
    c0 = -3.0
    gc: Dict[str, float] = {sc0: 1.0, sc1: 2.0}
    for bi in bnames:
        gc[bi] = gc.get(bi, 0.0) + 1.0
    add_square_of_affine(Qlin, Qquad, offset, c0, gc, weight=lam_card)

    # Optional numeric stabilization: rescale the BQM
    scale_applied = rescale_bqm_inplace(Qlin, Qquad, offset, target_max_abs=rescale_target)

    bqm = dimod.BinaryQuadraticModel(Qlin, Qquad, offset[0], vartype=dimod.BINARY)

    # ---- sample ----
    sampler = SimulatedAnnealingSampler()
    t0 = time.perf_counter()
    ss = sampler.sample(bqm, num_reads=sa.num_reads, sweeps=sa.sweeps, seed=sa.seed)
    t1 = time.perf_counter()

    best = ss.first.sample
    energy = float(ss.first.energy)

    # ---- decode ----
    x = {xi: decode_affine(best, cx[xi], ax[xi]) for xi in ["x1", "x2", "x3", "x4"]}
    b = {bi: int(best.get(bi, 0)) for bi in bnames}
    s = {si: decode_affine(best, cs[si], as_[si]) for si in ["s1", "s2", "s3", "s4"]}
    sc_val = int(best.get(sc0, 0)) + 2 * int(best.get(sc1, 0))

    # objective value
    f = (
        4.0 * x["x1"] ** 2
        + 6.0 * x["x1"] * x["x2"]
        - 2.0 * x["x1"] * x["x3"]
        + 6.0 * x["x2"] ** 2
        + 2.0 * x["x2"] * x["x3"]
        + 10.0 * x["x3"] ** 2
    )

    # residuals / violations (original-scale diagnostics)
    e1 = (x["x1"] + x["x2"] + x["x3"] + x["x4"] - 1.0)
    e2 = (8.0 * x["x1"] + 9.0 * x["x2"] + 12.0 * x["x3"] + 7.0 * x["x4"] - 10.0)

    # inequality violations (for reporting)
    linkV = max(0.0, x["x1"] - b["b6"], x["x2"] - b["b7"], x["x3"] - b["b8"], x["x4"] - b["b9"])
    cardV = max(0.0, (b["b6"] + b["b7"] + b["b8"] + b["b9"]) - 3)

    # equality residuals used in penalties (should be ~0 if penalty does its job)
    linkR = max(
        abs(x["x1"] - b["b6"] + s["s1"]),
        abs(x["x2"] - b["b7"] + s["s2"]),
        abs(x["x3"] - b["b8"] + s["s3"]),
        abs(x["x4"] - b["b9"] + s["s4"]),
    )
    cardR = (b["b6"] + b["b7"] + b["b8"] + b["b9"]) + sc_val - 3

    # a simple feasibility metric (lexicographic acceptance uses this)
    feas = max(abs(e1), abs(e2), linkV, cardV)

    return {
        "x": x,
        "b": b,
        "s": s,
        "sc": sc_val,
        "obj": f,
        "penE": energy,  # NOTE: this is energy after optional rescaling; only for sampler diagnostic
        "feas": feas,
        "e1": e1,
        "e2": e2,
        "linkV": linkV,
        "linkR": linkR,
        "cardV": cardV,
        "cardR": cardR,
        "stats": bqm_stats(bqm),
        "time_s": (t1 - t0),
        "lam_build": {"e1": lam_e1, "e2": lam_e2, "link": lam_link, "card": lam_card},
        "bqm_scale_applied": scale_applied,
        "sampleset": ss,  # kept in case you want alternate centers
    }


# ----------------------------
# Constant-size zoom + backtrack
# ----------------------------

@dataclass
class ZoomConfig:
    rho: float = 0.2                  # shrink factor per accepted zoom
    min_width_x: float = 1e-4         # absolute min width for x_i boxes
    min_width_s: float = 1e-4         # absolute min width for slack boxes
    max_iters: int = 25
    no_improve_backtracks: int = 2    # how many failed zoom rounds before stopping if stack empty

    # acceptance thresholds
    feas_tol: float = 5e-3            # “feasible enough” threshold for equalities (and inequalities)
    feas_eps: float = 1e-6            # strict improvement in feasibility
    obj_eps: float = 1e-8             # strict improvement in objective once feasible enough


def anchored_shrink_bounds(
    L0: float,
    U0: float,
    L: float,
    U: float,
    x_val: float,
    rho: float,
    min_width: float,
) -> Tuple[float, float]:
    """
    Shrink [L,U] by rho, but *anchor* the incumbent's normalized coordinate xhat so that the
    incumbent remains exactly representable on the new grid (when not clipped).
    This avoids the common “zoom but lose incumbent” issue in binary encodings.
    """
    width = U - L
    width_new = max(min_width, rho * width)

    # keep xhat from the current box (so same bit-pattern remains valid if no clipping)
    xhat = normalized_coord(x_val, L, U)

    # anchored new bounds: x_val = L_new + width_new * xhat
    L_new = x_val - width_new * xhat
    U_new = L_new + width_new

    # clip into original bounds [L0,U0] while preserving width_new
    if L_new < L0:
        L_new = L0
        U_new = L0 + width_new
    if U_new > U0:
        U_new = U0
        L_new = U0 - width_new

    # final safety
    if L_new < L0:
        L_new = L0
    if U_new > U0:
        U_new = U0
    if U_new < L_new:
        U_new = L_new

    return (L_new, U_new)


def better_solution(a: Dict[str, Any], b: Dict[str, Any], cfg: ZoomConfig) -> bool:
    """
    Return True if a is better than b (lexicographic):
      1) reduce feasibility metric
      2) if both feasible enough, reduce objective
    """
    if b is None:
        return True

    fa, fb = a["feas"], b["feas"]
    oa, ob = a["obj"], b["obj"]

    # strong feasibility improvement always wins
    if fa < fb - cfg.feas_eps:
        return True

    # if both are "feasible enough", compare objective
    if fa <= cfg.feas_tol and fb <= cfg.feas_tol:
        if oa < ob - cfg.obj_eps:
            return True

    return False


def rolling_zoom_constant_size_with_backtracking(
    Jx: int = 1,
    Js: int = 1,
    sa: SAOptions = SAOptions(),
    penalties: PenaltyConfig = PenaltyConfig(),
    zoom: ZoomConfig = ZoomConfig(),
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Constant-size zooming rolling precision:
      - Solve at current box
      - Propose zoomed boxes (x only, s only, x+s), accept first that improves incumbent
      - If none improves, backtrack to previous box (stack) if possible; else stop.
    """

    # Original bounds
    x0 = {f"x{i}": (0.0, 1.0) for i in range(1, 5)}
    s0 = {f"s{i}": (0.0, 1.0) for i in range(1, 5)}

    # start box
    box = Box(x_bounds=dict(x0), s_bounds=dict(s0))
    stack: List[Box] = []
    history: List[Dict[str, Any]] = []

    incumbent: Optional[Dict[str, Any]] = None
    fail_rounds = 0

    for it in range(zoom.max_iters):
        # Solve at current box
        sol = solve_alan_at_box(box, Jx, Js, penalties, sa)
        sol["iter"] = it
        sol["move"] = "solve"
        sol["box"] = {"x": dict(box.x_bounds), "s": dict(box.s_bounds)}

        # Update incumbent if improved
        if better_solution(sol, incumbent, zoom):
            incumbent = sol

        history.append(sol)

        if verbose:
            st = sol["stats"]
            xv = sol["x"]
            bv = sol["b"]
            print(
                f"it={it:02d} move=solve "
                f"x=[{xv['x1']:.6f},{xv['x2']:.6f},{xv['x3']:.6f},{xv['x4']:.6f}] "
                f"b=[{bv['b6']},{bv['b7']},{bv['b8']},{bv['b9']}] "
                f"obj={sol['obj']:.6e} feas={sol['feas']:.2e} "
                f"e1={sol['e1']:+.1e} e2={sol['e2']:+.1e} "
                f"linkV={sol['linkV']:.1e} cardV={sol['cardV']:.1e} "
                f"nvars={st['n_vars']} nquad={st['n_quad']} time={sol['time_s']:.3f}s"
            )

        # stopping if box already tiny and feasible enough
        max_wx = max(U - L for (L, U) in box.x_bounds.values())
        max_ws = max(U - L for (L, U) in box.s_bounds.values())
        if incumbent is not None and incumbent["feas"] <= zoom.feas_tol and max_wx <= zoom.min_width_x and max_ws <= zoom.min_width_s:
            break

        # Propose zoom moves around CURRENT incumbent (not just current box solve)
        if incumbent is None:
            break

        cx = incumbent["x"]
        cs = incumbent["s"]

        # Candidate boxes (try strongest first)
        candidates: List[Tuple[str, Box]] = []

        # helper to make a zoomed box
        def make_zoomed_box(zoom_x: bool, zoom_s: bool) -> Box:
            new_x = dict(box.x_bounds)
            new_s = dict(box.s_bounds)

            if zoom_x:
                for xi in ["x1", "x2", "x3", "x4"]:
                    L0, U0 = x0[xi]
                    L, U = box.x_bounds[xi]
                    new_x[xi] = anchored_shrink_bounds(L0, U0, L, U, cx[xi], zoom.rho, zoom.min_width_x)

            if zoom_s:
                for si in ["s1", "s2", "s3", "s4"]:
                    L0, U0 = s0[si]
                    L, U = box.s_bounds[si]
                    new_s[si] = anchored_shrink_bounds(L0, U0, L, U, cs[si], zoom.rho, zoom.min_width_s)

            return Box(x_bounds=new_x, s_bounds=new_s)

        candidates.append(("zoom_xs", make_zoomed_box(True, True)))
        candidates.append(("zoom_x", make_zoomed_box(True, False)))
        candidates.append(("zoom_s", make_zoomed_box(False, True)))

        accepted = False
        for move_name, cand_box in candidates:
            cand_sol = solve_alan_at_box(cand_box, Jx, Js, penalties, sa)
            cand_sol["iter"] = it
            cand_sol["move"] = move_name
            cand_sol["box"] = {"x": dict(cand_box.x_bounds), "s": dict(cand_box.s_bounds)}
            history.append(cand_sol)

            if better_solution(cand_sol, incumbent, zoom):
                # accept zoom: push current box on stack, move to new box
                stack.append(box)
                box = cand_box
                incumbent = cand_sol
                accepted = True
                fail_rounds = 0

                if verbose:
                    st = cand_sol["stats"]
                    xv = cand_sol["x"]
                    bv = cand_sol["b"]
                    print(
                        f"    ACCEPT {move_name:<7} "
                        f"x=[{xv['x1']:.6f},{xv['x2']:.6f},{xv['x3']:.6f},{xv['x4']:.6f}] "
                        f"b=[{bv['b6']},{bv['b7']},{bv['b8']},{bv['b9']}] "
                        f"obj={cand_sol['obj']:.6e} feas={cand_sol['feas']:.2e} "
                        f"nvars={st['n_vars']} nquad={st['n_quad']} time={cand_sol['time_s']:.3f}s"
                    )
                break

        if accepted:
            continue

        # If no zoom improved, try backtracking
        fail_rounds += 1
        if stack:
            box = stack.pop()
            if verbose:
                print("    BACKTRACK to previous box")
            continue

        # no stack to backtrack to
        if fail_rounds >= zoom.no_improve_backtracks:
            break

    # baseline “no zoom” solve at original box (for comparison)
    baseline_box = Box(x_bounds=dict(x0), s_bounds=dict(s0))
    baseline = solve_alan_at_box(baseline_box, Jx, Js, penalties, sa)

    return {
        "incumbent": incumbent,
        "history": history,
        "baseline_no_zoom": baseline,
        "Jx": Jx,
        "Js": Js,
    }


# ----------------------------
# Run
# ----------------------------

if __name__ == "__main__":
    print("\n--- Constant-size zoom-in rolling precision (alan) ---")
    report = rolling_zoom_constant_size_with_backtracking(
        Jx=1,  # keep small; precision comes from zooming bounds
        Js=1,
        sa=SAOptions(num_reads=300, sweeps=4000, seed=13),
        penalties=PenaltyConfig(
            lam_e1=200.0, lam_e2=200.0, lam_link=300.0, lam_card=200.0,
            dynamic=True, lam_cap=1e6
        ),
        zoom=ZoomConfig(
            rho=0.2,
            min_width_x=1e-4,
            min_width_s=1e-4,
            max_iters=20,
            feas_tol=5e-3
        ),
        verbose=True,
    )

    inc = report["incumbent"]
    base = report["baseline_no_zoom"]

    def fmt_vec_x(sol):
        x = sol["x"]
        return f"[{x['x1']:.6f},{x['x2']:.6f},{x['x3']:.6f},{x['x4']:.6f}]"

    def fmt_vec_b(sol):
        b = sol["b"]
        return f"[{b['b6']},{b['b7']},{b['b8']},{b['b9']}]"

    print("\n--- Baseline (no zoom, same constant QUBO size) ---")
    st = base["stats"]
    print(
        f"x={fmt_vec_x(base)} b={fmt_vec_b(base)} "
        f"obj={base['obj']:.6e} feas={base['feas']:.2e} "
        f"e1={base['e1']:+.1e} e2={base['e2']:+.1e} "
        f"linkV={base['linkV']:.1e} cardV={base['cardV']:.1e} "
        f"nvars={st['n_vars']} nquad={st['n_quad']} time={base['time_s']:.3f}s"
    )

    if inc is not None:
        print("\n--- Best found by zoom-in + backtracking ---")
        st = inc["stats"]
        print(
            f"x={fmt_vec_x(inc)} b={fmt_vec_b(inc)} "
            f"obj={inc['obj']:.6e} feas={inc['feas']:.2e} "
            f"e1={inc['e1']:+.1e} e2={inc['e2']:+.1e} "
            f"linkV={inc['linkV']:.1e} cardV={inc['cardV']:.1e} "
            f"nvars={st['n_vars']} nquad={st['n_quad']} time={inc['time_s']:.3f}s"
        )
    else:
        print("\nNo incumbent solution recorded (unexpected).")



--- Constant-size zoom-in rolling precision (alan) ---
it=00 move=solve x=[0.500000,0.000000,0.500000,0.000000] b=[1,0,1,1] obj=3.000000e+00 feas=0.00e+00 e1=+0.0e+00 e2=+0.0e+00 linkV=0.0e+00 cardV=0.0e+00 nvars=46 nquad=385 time=0.137s
    ACCEPT zoom_xs x=[0.400000,0.000000,0.520000,0.080000] b=[1,0,1,1] obj=2.928000e+00 feas=1.78e-15 nvars=46 nquad=385 time=0.131s
it=01 move=solve x=[0.400000,0.000000,0.520000,0.080000] b=[1,0,1,1] obj=2.928000e+00 feas=1.78e-15 e1=+2.2e-16 e2=+1.8e-15 linkV=0.0e+00 cardV=0.0e+00 nvars=46 nquad=385 time=0.130s
    BACKTRACK to previous box
it=02 move=solve x=[0.500000,0.000000,0.500000,0.000000] b=[1,0,1,1] obj=3.000000e+00 feas=0.00e+00 e1=+0.0e+00 e2=+0.0e+00 linkV=0.0e+00 cardV=0.0e+00 nvars=46 nquad=385 time=0.127s

--- Baseline (no zoom, same constant QUBO size) ---
x=[0.500000,0.000000,0.500000,0.000000] b=[1,0,1,1] obj=3.000000e+00 feas=0.00e+00 e1=+0.0e+00 e2=+0.0e+00 linkV=0.0e+00 cardV=0.0e+00 nvars=46 nquad=385 time=0.128s

--- Best fou